# Script 19 — Sensitivity models in Mplus: functional form, survey cycle, cognitive selection (Supp Tables S21–S23)

Three bounded sets of complex-survey sensitivity models, each fixed before it
is fitted and each reported next to the primary estimate it qualifies:

| table | question | inputs written by | report written by |
|---|---|---|---|
| **S21** functional form, age interaction, education coding | Do the three non-linear natural-spline terms add to the straight line in the rhythm → KDM-BA equation (3-df Wald)? Does age modify the association once the shape is free (4-df / 3-df)? Do quintile indicators with common cut-points interact with age? Do four education indicators change the nine primary models? | `9_mplus_models/build_p1.py` → `nhanes_sem_p1.dat`, `p1_*.inp`, `m_*_edud.inp` | `p1_report.py` |
| **S22** survey-cycle sensitivity | Does a 2011–12 / 2013–14 period term change any primary, age-interaction or P1 estimate (42 cycle-adjusted models)? Do the rhythm → biological-measure slopes differ between cycles (12 pooled single-outcome contrasts, 1-df Wald; 12 cycle-specific age models)? Motivated by the CDC CBC_H note on the mid-2012 instrument transition for RDW and WBC, both KDM inputs. | `build_cycle_audit.py` → `nhanes_sem_cycle.dat`, `9_mplus_models/cycle_audit/`; `build_cycle_contrasts.py` → `9_mplus_models/cycle_contrasts/` | `freeze_sensitivity_report.py` |
| **S23** cognitive availability and complete-case analysis | Who among examined adults 60+ contributed 0, 1, 2 or 3 cognitive scores (weighted profiles)? Do the three cognition models change when every modelled variable must be observed (complete case against FIML)? | `build_selection_audit.py` → `nhanes_sem_selection.dat`, `9_mplus_models/selection_audit/` | `freeze_sensitivity_report.py` |

Every Mplus run goes through `9_mplus_models/mplus_run.py`, which writes a
`.run.json` record binding input, data and output by sha256 and accepts an
output only if `mplus_output_qc.validate_output` passes (normal termination,
trustworthy standard errors, input echo identical to the input, expected
Wald df, positive cognition loadings). The reports read outputs only through
`mplus_output_qc.read_verified`, so a stale or failed run cannot enter a
table. The 18 split-cycle full SEMs (`cycle_audit/cycleG_m_*`, `cycleH_m_*`)
are generated for the audit record but retired from inference (eight carry
design-rank warnings); they are not run here and the report skips them.

All three sets were added during manuscript revision and are reported as
sensitivity analyses.

Run order: Step 1 → Mplus (Step 2, outside Jupyter) → Steps 3–4.


## Step 0 — Imports, paths

In [ ]:
import os, subprocess, sys
from pathlib import Path
import pandas as pd
pd.options.display.width = 220

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError('cannot find the project root from ' + str(Path.cwd())
                       + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE = find_root()
SCRIPTS = BASE / 'scripts'
MODELS = SCRIPTS / '9_mplus_models'
SUPP = BASE / 'result' / 'supplementary'
sys.path.insert(0, str(SCRIPTS))

def run(script, *args, cwd=SCRIPTS):
    """Run a project script in a subprocess and echo its output; stop on error."""
    cmd = [sys.executable, str(script), *args]
    print('$', ' '.join(str(c) for c in cmd[1:]))
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode:
        print(r.stderr[-3000:])
        raise RuntimeError(f'{Path(script).name} failed (exit {r.returncode})')
    return r.stdout

print('project root:', BASE)


## Step 1 — Write the data files and the Mplus inputs (before Mplus)

`build_p1.py` must run first: the cycle and selection sets extend its data file
(`nhanes_sem_p1.dat`) and edit its inputs. Each builder writes its plan
(`plan.json` in the model folder and under `result/qc/freeze/`).

In [ ]:
run(MODELS / 'build_p1.py')
run(SCRIPTS / 'build_cycle_audit.py')
run(SCRIPTS / 'build_cycle_contrasts.py')
run(SCRIPTS / 'build_selection_audit.py')
run(MODELS / 'build_p1.py', '--check')

inventory = {
    'p1_*.inp (S21)': len(list(MODELS.glob('p1_*.inp'))),
    'm_*_edud.inp (S21)': len(list(MODELS.glob('m_*_edud.inp'))),
    'cycle_audit/adjusted_*.inp (S22)': len(list((MODELS / 'cycle_audit').glob('adjusted_*.inp'))),
    'cycle_audit/cycle?_x_age_*.inp (S22)': len(list((MODELS / 'cycle_audit').glob('cycle?_x_age_*.inp'))),
    'cycle_audit/cycle?_m_*.inp (retired, not run)': len(list((MODELS / 'cycle_audit').glob('cycle?_m_*.inp'))),
    'cycle_contrasts/*.inp (S22)': len(list((MODELS / 'cycle_contrasts').glob('*.inp'))),
    'selection_audit/*.inp (S23)': len(list((MODELS / 'selection_audit').glob('*.inp'))),
}
print(pd.Series(inventory).to_string())
assert inventory['p1_*.inp (S21)'] == 12 and inventory['m_*_edud.inp (S21)'] == 9
assert inventory['cycle_audit/adjusted_*.inp (S22)'] == 42 and inventory['cycle_audit/cycle?_x_age_*.inp (S22)'] == 12
assert inventory['cycle_contrasts/*.inp (S22)'] == 12 and inventory['selection_audit/*.inp (S23)'] == 3


## Step 2 — Run the models in Mplus (outside Jupyter)

From `scripts/9_mplus_models`, with Mplus 9 installed (set `MPLUS` if it is not
on the PATH). 90 models in all; the retired split-cycle full SEMs are not run.

```bash
cd scripts/9_mplus_models
python mplus_run.py --record '*.inp'               # bind the 51 baseline outputs of the
                                                   # unified re-run (no Mplus run; see below)
python mplus_run.py 'p1_*.inp'                     # 12  S21
python mplus_run.py '*_edud.inp'                   #  9  S21
python mplus_run.py 'cycle_audit/adjusted_*.inp'   # 42  S22
python mplus_run.py 'cycle_audit/cycle?_x_age_*.inp'   # 12  S22
python mplus_run.py 'cycle_contrasts/*.inp'        # 12  S22
python mplus_run.py 'selection_audit/*.inp'        #  3  S23
```

`mplus_run.py` stops at the first model that fails its gate and leaves the
output as `.failed.out`; nothing downstream reads a failed run.

**Baseline records.** `freeze_sensitivity_report.py` compares every variant
with its baseline model (`m_*`, `m_*_vol`, `x_age_*`, `p1_*`) through
`read_verified`, so the baseline outputs need a `.run.json` as well. The
baseline models of the unified re-run were produced by `run_all.sh`, which
writes no record; `mplus_run.py --record` validates each existing output
(`validate_output`: normal termination, trustworthy standard errors, input
echo identical to the input file) and writes the record with
`recorded_post_hoc = true` and the run date printed in the Mplus header. It
never runs Mplus and never changes an output.


## Step 3 — Reports (after Mplus): S21, S22, S23

In [ ]:
import json

def recorded(folder, pattern):
    """Inputs matching the pattern whose output has a PASS run record."""
    out = []
    for inp in sorted((MODELS / folder).glob(pattern)):
        rec = inp.with_suffix('.run.json')
        if inp.with_suffix('.out').exists() and rec.exists() and json.loads(rec.read_text()).get('status') == 'PASS':
            out.append(inp.stem)
    return out

def missing(folder, pattern, n):
    have = recorded(folder, pattern)
    want = sorted(p.stem for p in (MODELS / folder).glob(pattern))
    return [m for m in want if m not in have] if len(have) < n else []

# baseline models the cycle and selection comparisons read through read_verified
cycle_plan = json.loads((MODELS / 'cycle_audit' / 'plan.json').read_text())
retired = set(json.loads((MODELS / 'cycle_contrasts' / 'plan.json').read_text())['retired_models'])
bases = sorted({r['base'] for r in cycle_plan['models'] if r['model'] not in retired}
               | {r['base'] for r in json.loads((MODELS / 'selection_audit' / 'plan.json').read_text())['models']})
base_missing = [b for b in bases if b not in recorded('', b + '.inp')]

pending = {
    'S21': missing('', 'p1_*.inp', 12) + missing('', 'm_*_edud.inp', 9),
    'S22': missing('cycle_audit', 'adjusted_*.inp', 42) + missing('cycle_audit', 'cycle?_x_age_*.inp', 12)
           + missing('cycle_contrasts', '*.inp', 12),
    'S23': missing('selection_audit', '*.inp', 3),
}
for k, v in pending.items():
    print(f'{k}: ' + ('ready' if not v else f'{len(v)} models without a PASS record, e.g. {v[:3]}'))
baseline_pending = [b for b in base_missing if not b.startswith('p1_')]   # p1_* are counted under S21
if baseline_pending:
    print(f'baseline records missing for {len(baseline_pending)} models, e.g. {baseline_pending[:3]} -> '
          "run:  python mplus_run.py --record '*.inp'   (in scripts/9_mplus_models)")

if not pending['S21']:
    run(SCRIPTS / 'p1_report.py')
else:
    print('S21 pending: run the p1_*.inp and *_edud.inp models with mplus_run.py (Step 2)')
if not (pending['S22'] or pending['S23'] or base_missing):
    run(SCRIPTS / 'freeze_sensitivity_report.py')
else:
    print('S22/S23 pending: run the cycle_audit, cycle_contrasts and selection_audit models with mplus_run.py (Step 2)')


## Step 4 — Registry, checkers, supplement

`captions.py` and `display_map.py` register S21–S23 as soon as their workbooks
exist; `build_supplement_word.py` then adds them to the Supplementary Materials
file. Nothing here changes a primary result.

In [ ]:
for s in ('display_map.py', 'captions.py', 'variable_dictionary.py'):
    run(SCRIPTS / s, '--write')
for stem, label in [('SuppTableS21_P1_FunctionalForm', 'S21'), ('SuppTableS22_CycleSensitivity', 'S22'), ('SuppTableS23_CognitiveSelection', 'S23')]:
    f = SUPP / f'{stem}.xlsx'
    print(f'{label}: ' + (f'{f.name} ({len(pd.ExcelFile(f).sheet_names)} sheets)' if f.exists() else 'not yet written'))
print('\nRebuild the Supplementary Materials with:  python scripts/build_supplement_word.py')
